# 🧠 DL Uplift Benchmark — TARNet · CFR · DragonNet · RERUM

> **Цель:** протестировать DL-подходы к uplift прибыли (`rec_spend`).
> Обучение на **MPS** (Apple Silicon). Метрика: `uplift@10` бинарный (buy/no-buy) — идентично `pilot_modeling.ipynb`.
>
> **🐛 Баг коллеги:** TARNet/DragonNet считались через `uplift_at_k_manual` (непрерывный ₽≈16),
> а RERUM через sklift (бинарный≈0.03). **Разные шкалы — нельзя сравнивать!** Здесь всё единообразно.


## 0. Установка библиотек

In [ ]:
%%bash
pip install torch torchvision torchaudio -q
pip install lightgbm scikit-uplift pandas pyarrow scikit-learn numpy scipy -q


## 1. Импорты и конфиг

In [ ]:
import warnings, json, time, os, math
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklift.metrics import uplift_at_k, qini_auc_score, uplift_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ─── Device: MPS (Apple Silicon) → CUDA → CPU ────────────────────────────────
def get_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device('mps')
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')

DEVICE = get_device()

# ─── Constants ────────────────────────────────────────────────────────────────
RANDOM_STATE  = 42
N_FOLDS       = 5
ARTIFACTS_DIR = 'dl_artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

TARGET_COL    = 'rec_spend'
TREATMENT_COL = 'treatment_flg'
ID_COL        = 'user_id'
COMM_COL      = 'communication_type'

# ─── Neural-net hyper-params ──────────────────────────────────────────────────
NN_HIDDEN_REP  = (256, 128)
NN_HIDDEN_HEAD = (64,)
NN_DROPOUT     = 0.2
NN_LR          = 1e-3
NN_BATCH_SIZE  = 1024
NN_EPOCHS      = 40
NN_WARMUP_EP   = 8        # RERUM: ranking losses начинают действовать после warmup

TOP25 = [
    'cus_cat_7_rto', 'n_redeem', 'cus_cat_7_atv', 'cus_cat_5_rto',
    'cus_cat_5_std', 'mntv', 'n_days_life', 'cus_cat_5_atv',
    'cus_cat_7_n_days_big_period', 'cus_cat_7_std', 'cus_mark_n_offers',
    'cus_cat_6_rto', 'rto_format_1', 'age', 'cus_mark_n_view', 'n_sku',
    'cus_cat_6_std', 'cus_cat_6_atv', 'cus_cat_5_max_min_days_diff',
    'cus_mark_n_rule', 'p_25_tv', 'cus_cat_7_max_min_days_diff',
    'mxtv', 'rto_format_3', 'avg_days_btw_trn'
]

def seed_everything(seed=RANDOM_STATE):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

seed_everything()
print(f'✅ Config loaded')
print(f'   Device  : {DEVICE}')
print(f'   PyTorch : {torch.__version__}')
print(f'   Artifacts: {ARTIFACTS_DIR}/')


## 2. Загрузка и подготовка данных

In [ ]:
train = pd.read_parquet('train.parquet')
test  = pd.read_parquet('test.parquet')
print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Treatment balance: T={train[TREATMENT_COL].mean():.3f}')

ALL_FEATS = [c for c in train.columns
             if c not in [ID_COL, TARGET_COL, TREATMENT_COL]]

le_map = {}
for col in train[ALL_FEATS].select_dtypes(include=['object','string','category']).columns:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], ignore_index=True).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))
    le_map[col] = le

print(f'Features: {len(ALL_FEATS)}')
print(f'Target zeros: {(train[TARGET_COL]==0).mean():.1%}')


## 3. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy(); eps = 1e-6
    df['cat7_share']         = df['cus_cat_7_rto'] / (df['rto'] + eps)
    df['cat6_share']         = df['cus_cat_6_rto'] / (df['rto'] + eps)
    df['cat5_share']         = df['cus_cat_5_rto'] / (df['rto'] + eps)
    df['mkt_resp_rate']      = df['cus_mark_n_rule'] / (df['cus_mark_n_offers'].clip(lower=1))
    df['mkt_view_rate']      = df['cus_mark_n_view'] / (df['cus_mark_n_offers'].clip(lower=1))
    df['recency_ratio']      = df['cus_cat_7_last_1_days'] / (df['avg_days_btw_trn'] + eps)
    df['spend_cv']           = df['stdtv'] / (df['atv'] + eps)
    df['trn_density']        = df['n_trn'] / (df['n_days_life'] + eps) * 365
    df['cat_breadth_ratio']  = df['n_cat_7'] / (df['n_cat_5'] + eps)
    df['cat7_vs_last_visit'] = df['cus_cat_7_last_1_days'] / (df['n_days_last_visit'] + eps)
    return df

train = engineer_features(train)
test  = engineer_features(test)

NEW_FEATS = [
    'cat7_share','cat6_share','cat5_share',
    'mkt_resp_rate','mkt_view_rate','recency_ratio',
    'spend_cv','trn_density','cat_breadth_ratio','cat7_vs_last_visit'
]

FEAT_COLS = ALL_FEATS + NEW_FEATS
print(f'Total features: {len(FEAT_COLS)}, incl. {len(NEW_FEATS)} engineered')


## 4. Утилиты валидации

**Единая метрика для всех моделей** — `uplift@10` (бинарный y = buy/no-buy) из scikit-uplift.
Это устраняет несопоставимость из ноута коллеги.


In [ ]:
def make_stratify_col(df):
    return df[TREATMENT_COL].astype(str) + '_' + df[COMM_COL].astype(str)

def oof_uplift_at_k(y_true, scores, treatment, k=0.1, n_boot=100, seed=42):
    y_bin = (y_true > 0).astype(int)
    point = uplift_at_k(y_bin, scores, treatment, strategy='overall', k=k)
    rng   = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        s = rng.choice(len(y_true), len(y_true), replace=True)
        try:
            boots.append(uplift_at_k(y_bin[s], scores[s], treatment[s], strategy='overall', k=k))
        except Exception:
            pass
    boots = np.array(boots)
    return float(point), float(np.percentile(boots, 10)), float(np.percentile(boots, 90))

def oof_qini(y_true, scores, treatment):
    return float(qini_auc_score((y_true > 0).astype(int), scores, treatment))

def get_folds(df, n_splits=N_FOLDS):
    strat = make_stratify_col(df)
    skf   = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    return list(skf.split(df, strat))

RESULTS = {}

def register_result(name, y_val, scores_val, treat_val, elapsed):
    u10, ci_lo, ci_hi = oof_uplift_at_k(y_val, scores_val, treat_val)
    qauc              = oof_qini(y_val, scores_val, treat_val)
    RESULTS[name]     = {'uplift@10': u10, 'ci80_lo': ci_lo, 'ci80_hi': ci_hi,
                         'qini_auc': qauc, 'elapsed_sec': round(elapsed, 1)}
    print(f'  [{name}] uplift@10={u10:.4f} [{ci_lo:.4f},{ci_hi:.4f}]'
          f'  Qini={qauc:.4f}  t={elapsed:.0f}s')
    return u10

folds = get_folds(train)
y     = train[TARGET_COL].values
T     = train[TREATMENT_COL].values
print('✅ Validation utils ready')


## 5. Общие блоки нейросетей

In [ ]:
class UpliftDataset(Dataset):
    def __init__(self, X, T, y):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.T = torch.as_tensor(T, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.T[idx], self.y[idx]


class MLP(nn.Module):
    """Linear → BatchNorm → ELU → Dropout (стек hidden слоёв)."""
    def __init__(self, in_dim, hidden_dims, dropout=0.2):
        super().__init__()
        layers = []; prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ELU(), nn.Dropout(dropout)]
            prev = h
        self.net = nn.Sequential(*layers)
        self.out_dim = prev
    def forward(self, x): return self.net(x)


def prepare_nn_features(df_tr, df_val, feat_cols):
    """StandardScaler fit только на train-фолде (предотвращает утечку)."""
    X_tr  = df_tr[feat_cols].fillna(-999).values.astype(np.float32)
    X_val = df_val[feat_cols].fillna(-999).values.astype(np.float32)
    sc    = StandardScaler()
    X_tr  = sc.fit_transform(X_tr)
    X_val = sc.transform(X_val)
    return X_tr.astype(np.float32), X_val.astype(np.float32)

def prepare_nn_test(df_tr, df_test, feat_cols):
    X_tr   = df_tr[feat_cols].fillna(-999).values.astype(np.float32)
    X_test = df_test[feat_cols].fillna(-999).values.astype(np.float32)
    sc     = StandardScaler()
    X_tr   = sc.fit_transform(X_tr)
    X_test = sc.transform(X_test)
    return X_tr.astype(np.float32), X_test.astype(np.float32)

def make_loader(X, T, y, shuffle=True):
    """drop_last=True при shuffle — критично для BatchNorm."""
    g = torch.Generator(); g.manual_seed(RANDOM_STATE)
    return DataLoader(UpliftDataset(X, T, y), batch_size=NN_BATCH_SIZE,
                      shuffle=shuffle, generator=g, drop_last=shuffle)

print('✅ NN utils ready')
print(f'   Training device: {DEVICE}')


## 6. Model A — TARNet

Treatment-Agnostic Representation Network (Shalit et al. 2017).
Shared encoder + два независимых регрессора для T=1 и T=0. Loss: MSE на `log1p(y)`.
Uplift = `expm1(ŷ₁) − expm1(ŷ₀)`.


In [ ]:
class TARNet(nn.Module):
    def __init__(self, in_dim, hidden_rep=NN_HIDDEN_REP, hidden_head=NN_HIDDEN_HEAD, dropout=NN_DROPOUT):
        super().__init__()
        self.rep   = MLP(in_dim, hidden_rep, dropout)
        self.head1 = nn.Sequential(MLP(self.rep.out_dim, hidden_head, dropout), nn.Linear(hidden_head[-1], 1))
        self.head0 = nn.Sequential(MLP(self.rep.out_dim, hidden_head, dropout), nn.Linear(hidden_head[-1], 1))
    def forward(self, x):
        h = self.rep(x)
        return self.head1(h).squeeze(-1), self.head0(h).squeeze(-1)

def train_tarnet(X_tr, T_tr, y_tr, in_dim, epochs=NN_EPOCHS):
    seed_everything()
    model = TARNet(in_dim).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=NN_LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    dl    = make_loader(X_tr, T_tr, np.log1p(y_tr))
    model.train()
    for ep in range(epochs):
        for xb, tb, yb in dl:
            xb, tb, yb = xb.to(DEVICE), tb.to(DEVICE), yb.to(DEVICE)
            y1, y0 = model(xb)
            m1, m0 = tb > 0.5, tb < 0.5
            loss = torch.tensor(0.0, device=DEVICE)
            if m1.any(): loss = loss + F.mse_loss(y1[m1], yb[m1])
            if m0.any(): loss = loss + F.mse_loss(y0[m0], yb[m0])
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    return model

@torch.no_grad()
def predict_tarnet(model, X_val):
    model.eval()
    Xb = torch.as_tensor(X_val, dtype=torch.float32).to(DEVICE)
    y1, y0 = model(Xb)
    return (torch.expm1(y1) - torch.expm1(y0)).cpu().numpy()

print('Training TARNet...')
t0 = time.time(); oof_scores_tar = np.zeros(len(train))
for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr_nn, X_val_nn = prepare_nn_features(train.iloc[tr_idx], train.iloc[val_idx], FEAT_COLS)
    model = train_tarnet(X_tr_nn, T[tr_idx].astype(np.float32),
                         y[tr_idx].astype(np.float32), in_dim=X_tr_nn.shape[1])
    oof_scores_tar[val_idx] = predict_tarnet(model, X_val_nn)
    print(f'  Fold {fold+1}/{N_FOLDS} done')
u10_tar = register_result('TARNet', y, oof_scores_tar, T, time.time()-t0)
np.save(f'{ARTIFACTS_DIR}/oof_tarnet.npy', oof_scores_tar)


## 7. Models B & C — CFR (Wasserstein и MMD)

CFR = TARNet + IPM-регуляризатор на расстояние между представлениями T=1 и T=0.
- **Wasserstein**: sliced Wasserstein-1 через 1D проекции
- **MMD**: Maximum Mean Discrepancy с RBF-ядром


In [ ]:
def wasserstein_loss(h1, h0, n_proj=16):
    d  = h1.shape[1]
    w  = F.normalize(torch.randn(d, n_proj, device=h1.device), dim=0)
    p1 = torch.sort(h1 @ w, dim=0).values
    p0 = torch.sort(h0 @ w, dim=0).values
    n  = min(p1.shape[0], p0.shape[0])
    return (p1[:n] - p0[:n]).abs().mean()

def mmd_loss(h1, h0, sigma=1.0):
    def rbf(a, b): return torch.exp(-torch.cdist(a, b)**2 / (2 * sigma**2))
    n1, n0 = h1.shape[0], h0.shape[0]
    return rbf(h1,h1).sum()/(n1*n1) + rbf(h0,h0).sum()/(n0*n0) - 2*rbf(h1,h0).sum()/(n1*n0)

class CFRNet(nn.Module):
    def __init__(self, in_dim, hidden_rep=NN_HIDDEN_REP, hidden_head=NN_HIDDEN_HEAD, dropout=NN_DROPOUT):
        super().__init__()
        self.rep   = MLP(in_dim, hidden_rep, dropout)
        self.head1 = nn.Sequential(MLP(self.rep.out_dim, hidden_head, dropout), nn.Linear(hidden_head[-1], 1))
        self.head0 = nn.Sequential(MLP(self.rep.out_dim, hidden_head, dropout), nn.Linear(hidden_head[-1], 1))
    def forward(self, x):
        h = self.rep(x)
        return self.head1(h).squeeze(-1), self.head0(h).squeeze(-1), h

def train_cfr(X_tr, T_tr, y_tr, in_dim, ipm='wass', alpha=0.5, epochs=NN_EPOCHS):
    seed_everything()
    model  = CFRNet(in_dim).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=NN_LR, weight_decay=1e-5)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    dl     = make_loader(X_tr, T_tr, np.log1p(y_tr))
    ipm_fn = wasserstein_loss if ipm == 'wass' else mmd_loss
    model.train()
    for ep in range(epochs):
        for xb, tb, yb in dl:
            xb, tb, yb = xb.to(DEVICE), tb.to(DEVICE), yb.to(DEVICE)
            y1, y0, h  = model(xb)
            m1, m0     = tb > 0.5, tb < 0.5
            loss_y = torch.tensor(0.0, device=DEVICE)
            if m1.any(): loss_y = loss_y + F.mse_loss(y1[m1], yb[m1])
            if m0.any(): loss_y = loss_y + F.mse_loss(y0[m0], yb[m0])
            loss_ipm = ipm_fn(h[m1], h[m0]) if (m1.any() and m0.any())                        else torch.tensor(0.0, device=DEVICE)
            loss = loss_y + alpha * loss_ipm
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    return model

@torch.no_grad()
def predict_cfr(model, X_val):
    model.eval()
    Xb = torch.as_tensor(X_val, dtype=torch.float32).to(DEVICE)
    y1, y0, _ = model(Xb)
    return (torch.expm1(y1) - torch.expm1(y0)).cpu().numpy()

# CFR Wasserstein
print('Training CFR (Wasserstein)...')
t0 = time.time(); oof_scores_cfr_wass = np.zeros(len(train))
for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr_nn, X_val_nn = prepare_nn_features(train.iloc[tr_idx], train.iloc[val_idx], FEAT_COLS)
    model = train_cfr(X_tr_nn, T[tr_idx].astype(np.float32),
                      y[tr_idx].astype(np.float32), in_dim=X_tr_nn.shape[1], ipm='wass')
    oof_scores_cfr_wass[val_idx] = predict_cfr(model, X_val_nn)
    print(f'  Fold {fold+1}/{N_FOLDS} done')
u10_cfr_wass = register_result('CFR (Wasserstein)', y, oof_scores_cfr_wass, T, time.time()-t0)
np.save(f'{ARTIFACTS_DIR}/oof_cfr_wass.npy', oof_scores_cfr_wass)


In [ ]:
# CFR MMD
print('Training CFR (MMD)...')
t0 = time.time(); oof_scores_cfr_mmd = np.zeros(len(train))
for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr_nn, X_val_nn = prepare_nn_features(train.iloc[tr_idx], train.iloc[val_idx], FEAT_COLS)
    model = train_cfr(X_tr_nn, T[tr_idx].astype(np.float32),
                      y[tr_idx].astype(np.float32), in_dim=X_tr_nn.shape[1], ipm='mmd')
    oof_scores_cfr_mmd[val_idx] = predict_cfr(model, X_val_nn)
    print(f'  Fold {fold+1}/{N_FOLDS} done')
u10_cfr_mmd = register_result('CFR (MMD)', y, oof_scores_cfr_mmd, T, time.time()-t0)
np.save(f'{ARTIFACTS_DIR}/oof_cfr_mmd.npy', oof_scores_cfr_mmd)


## 8. Model D — DragonNet

Shi et al. 2019. TARNet + голова propensity score (BCE).
Propensity loss регуляризует shared representation через достаточность propensity.


In [ ]:
class DragonNet(nn.Module):
    def __init__(self, in_dim, hidden_rep=NN_HIDDEN_REP, hidden_head=NN_HIDDEN_HEAD, dropout=NN_DROPOUT):
        super().__init__()
        self.rep    = MLP(in_dim, hidden_rep, dropout)
        self.head1  = nn.Sequential(MLP(self.rep.out_dim, hidden_head, dropout), nn.Linear(hidden_head[-1], 1))
        self.head0  = nn.Sequential(MLP(self.rep.out_dim, hidden_head, dropout), nn.Linear(hidden_head[-1], 1))
        self.head_t = nn.Sequential(nn.Linear(self.rep.out_dim, 32), nn.ELU(), nn.Linear(32, 1))
    def forward(self, x):
        h = self.rep(x)
        return self.head1(h).squeeze(-1), self.head0(h).squeeze(-1), self.head_t(h).squeeze(-1)

def train_dragonnet(X_tr, T_tr, y_tr, in_dim, epochs=NN_EPOCHS, alpha_prop=1.0):
    seed_everything()
    model = DragonNet(in_dim).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=NN_LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    dl    = make_loader(X_tr, T_tr, np.log1p(y_tr))
    bce   = nn.BCEWithLogitsLoss()
    model.train()
    for ep in range(epochs):
        for xb, tb, yb in dl:
            xb, tb, yb      = xb.to(DEVICE), tb.to(DEVICE), yb.to(DEVICE)
            y1, y0, t_logit = model(xb)
            m1, m0          = tb > 0.5, tb < 0.5
            loss_y = torch.tensor(0.0, device=DEVICE)
            if m1.any(): loss_y = loss_y + F.mse_loss(y1[m1], yb[m1])
            if m0.any(): loss_y = loss_y + F.mse_loss(y0[m0], yb[m0])
            loss = loss_y + alpha_prop * bce(t_logit, tb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    return model

@torch.no_grad()
def predict_dragonnet(model, X_val):
    model.eval()
    Xb = torch.as_tensor(X_val, dtype=torch.float32).to(DEVICE)
    y1, y0, _ = model(Xb)
    return (torch.expm1(y1) - torch.expm1(y0)).cpu().numpy()

print('Training DragonNet...')
t0 = time.time(); oof_scores_dragon = np.zeros(len(train))
for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr_nn, X_val_nn = prepare_nn_features(train.iloc[tr_idx], train.iloc[val_idx], FEAT_COLS)
    model = train_dragonnet(X_tr_nn, T[tr_idx].astype(np.float32),
                            y[tr_idx].astype(np.float32), in_dim=X_tr_nn.shape[1])
    oof_scores_dragon[val_idx] = predict_dragonnet(model, X_val_nn)
    print(f'  Fold {fold+1}/{N_FOLDS} done')
u10_dragon = register_result('DragonNet', y, oof_scores_dragon, T, time.time()-t0)
np.save(f'{ARTIFACTS_DIR}/oof_dragonnet.npy', oof_scores_dragon)


## 9. Model E — RERUM (KDD'24, arxiv 2405.15301)

**Rankability-enhanced Revenue Uplift Modeling Framework**.
Надстройка над DragonNet с тремя компонентами:

### Компоненты RERUM:
1. **ZILN loss** (Zero-Inflated Log-Normal) вместо MSE:
   - Голова: `(logit_p, μ, log_σ)` → `E[Y] = sigmoid(logit_p) · exp(μ + σ²/2)`
   - Отдельно моделирует P(покупка) и E[spend | покупка] — идеально для ~90% нулей
2. **Response Ranking Loss** — пары (i,j) внутри T=1/T=0:
   - Штраф если `y_i > y_j` но `ŷ_i < ŷ_j`
3. **Listwise Uplift Ranking Loss** (ListNet):
   - `L = −Σᵢ τᵢ · log softmax(τ̂ᵢ)`, где τ — псевдо-uplift
4. **Uncertainty Weighting** — 4 learnable `log_vars` балансируют лоссы автоматически

Ranking losses включаются после `NN_WARMUP_EP` эпох (ZILN сначала стабилизирует веса).


In [ ]:
# ─── ZILN loss ────────────────────────────────────────────────────────────────
def ziln_loss(params, y):
    """Zero-Inflated Log-Normal NLL. params: (B,3) = [logit_p, mu, log_sigma]"""
    logit_p, mu, log_sigma = params[:,0], params[:,1], params[:,2]
    sigma   = torch.clamp(F.softplus(log_sigma) + 1e-3, max=2.0)
    mu      = torch.clamp(mu, min=-5.0, max=10.0)
    is_zero = (y <= 0).float()
    log_1mp = -F.softplus(logit_p)
    log_p   = -F.softplus(-logit_p)
    y_safe  = y.clamp(min=1e-3)
    log_y   = torch.log(y_safe)
    lognorm_nll = (0.5*math.log(2*math.pi) + torch.log(sigma) + log_y
                   + 0.5*((log_y - mu)/sigma)**2)
    ll = is_zero * log_1mp + (1 - is_zero) * (log_p - lognorm_nll)
    return -ll.mean()

def expected_ziln(params):
    """E[Y] = p * exp(mu + sigma^2/2)"""
    logit_p, mu, log_sigma = params[:,0], params[:,1], params[:,2]
    p     = torch.sigmoid(logit_p)
    sigma = torch.clamp(F.softplus(log_sigma) + 1e-3, max=2.0)
    mu    = torch.clamp(mu, min=-5.0, max=10.0)
    return p * torch.exp(mu + 0.5 * sigma**2)

# ─── Response Ranking Loss ────────────────────────────────────────────────────
def response_ranking_loss(pred, y_real, n_pairs=1024):
    n = pred.size(0)
    if n < 2: return torch.tensor(0.0, device=pred.device)
    np_ = min(n_pairs, n * 2)
    ia  = torch.randint(0, n, (np_,), device=pred.device)
    ib  = torch.randint(0, n, (np_,), device=pred.device)
    yd  = y_real[ia] - y_real[ib]
    pd_ = pred[ia]   - pred[ib]
    wrong = ((yd > 1e-6) & (pd_ < 0)) | ((yd < -1e-6) & (pd_ > 0))
    if not wrong.any(): return torch.tensor(0.0, device=pred.device)
    return (pd_[wrong] - yd[wrong]).pow(2).mean()

# ─── Listwise Uplift Ranking Loss (ListNet formulation) ───────────────────────
def listwise_uplift_loss(uplift_hat, T, y, temperature=2.0):
    y_bin        = (y > 0).float()
    pseudo       = T * y_bin - (1 - T) * y_bin
    log_probs    = F.log_softmax(uplift_hat / temperature, dim=0)
    target_probs = F.softmax(pseudo, dim=0)
    return -(target_probs * log_probs).sum()

# ─── RERUM model ──────────────────────────────────────────────────────────────
class RERUM(nn.Module):
    def __init__(self, in_dim, hidden_rep=NN_HIDDEN_REP, hidden_head=NN_HIDDEN_HEAD, dropout=NN_DROPOUT):
        super().__init__()
        self.rep      = MLP(in_dim, hidden_rep, dropout)
        self.head1    = MLP(self.rep.out_dim, hidden_head, dropout)
        self.head0    = MLP(self.rep.out_dim, hidden_head, dropout)
        self.out1     = nn.Linear(hidden_head[-1], 3)  # ZILN params
        self.out0     = nn.Linear(hidden_head[-1], 3)
        self.head_t   = nn.Sequential(nn.Linear(self.rep.out_dim, 32), nn.ELU(), nn.Linear(32, 1))
        self.log_vars = nn.Parameter(torch.zeros(4))   # uncertainty weighting

    def forward(self, x):
        h = self.rep(x)
        p1, p0 = self.out1(self.head1(h)), self.out0(self.head0(h))
        return p1, p0, expected_ziln(p1), expected_ziln(p0), self.head_t(h).squeeze(-1)

def train_rerum(X_tr, T_tr, y_tr, in_dim, epochs=NN_EPOCHS, warmup=NN_WARMUP_EP):
    seed_everything()
    model = RERUM(in_dim).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=NN_LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    dl    = make_loader(X_tr, T_tr, y_tr)   # raw y — ZILN работает с оригинальными значениями
    bce   = nn.BCEWithLogitsLoss()
    model.train()
    for ep in range(epochs):
        ranking_on = (ep >= warmup)
        for xb, tb, yb in dl:
            xb, tb, yb                 = xb.to(DEVICE), tb.to(DEVICE), yb.to(DEVICE)
            p1, p0, ey1, ey0, t_logit  = model(xb)
            uplift                     = ey1 - ey0
            m1, m0                     = tb > 0.5, tb < 0.5

            lz_t = ziln_loss(p1[m1], yb[m1]) if m1.any() else torch.tensor(0., device=DEVICE)
            lz_c = ziln_loss(p0[m0], yb[m0]) if m0.any() else torch.tensor(0., device=DEVICE)

            lr_ = torch.tensor(0., device=DEVICE)
            if ranking_on:
                if m1.sum() >= 2: lr_ = lr_ + response_ranking_loss(ey1[m1], yb[m1])
                if m0.sum() >= 2: lr_ = lr_ + response_ranking_loss(ey0[m0], yb[m0])

            lu_ = listwise_uplift_loss(uplift, tb, yb)                   if (ranking_on and xb.size(0) >= 32) else torch.tensor(0., device=DEVICE)

            losses    = torch.stack([lz_t, lz_c, lr_, lu_])
            precision = torch.exp(-model.log_vars)
            loss      = (precision * losses).sum() + model.log_vars.sum()
            loss      = loss + bce(t_logit, tb)

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
    return model

@torch.no_grad()
def predict_rerum(model, X_val):
    model.eval()
    Xb = torch.as_tensor(X_val, dtype=torch.float32).to(DEVICE)
    _, _, ey1, ey0, _ = model(Xb)
    return (ey1 - ey0).cpu().numpy()


print('Training RERUM (DragonNet + ZILN + Ranking losses)...')
t0 = time.time(); oof_scores_rerum = np.zeros(len(train))
for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr_nn, X_val_nn = prepare_nn_features(train.iloc[tr_idx], train.iloc[val_idx], FEAT_COLS)
    model = train_rerum(X_tr_nn, T[tr_idx].astype(np.float32),
                        y[tr_idx].astype(np.float32), in_dim=X_tr_nn.shape[1])
    oof_scores_rerum[val_idx] = predict_rerum(model, X_val_nn)
    print(f'  Fold {fold+1}/{N_FOLDS} done')
u10_rerum = register_result('RERUM (DragonNet)', y, oof_scores_rerum, T, time.time()-t0)
np.save(f'{ARTIFACTS_DIR}/oof_rerum.npy', oof_scores_rerum)


## 10. Сводная таблица метрик

In [ ]:
# Подтягиваем LGB-baseline из pilot_modeling.ipynb (если есть)
for name, path in [('Optuna Hurdle (LGB)', 'pilot_artifacts/oof_optuna.npy'),
                   ('X-Learner (LGB)',     'pilot_artifacts/oof_xlearner.npy')]:
    if os.path.exists(path):
        sc = np.load(path)
        if len(sc) == len(y):
            u10, lo, hi = oof_uplift_at_k(y, sc, T)
            RESULTS[name] = {'uplift@10': u10, 'ci80_lo': lo, 'ci80_hi': hi,
                             'qini_auc': oof_qini(y, sc, T), 'elapsed_sec': 0.0}
            print(f'  Loaded baseline [{name}]: uplift@10={u10:.4f}')

results_df = pd.DataFrame([
    {'model': k, 'uplift@10': v['uplift@10'], 'ci80_lo': v['ci80_lo'],
     'ci80_hi': v['ci80_hi'], 'qini_auc': v['qini_auc'], 'elapsed_sec': v['elapsed_sec']}
    for k, v in RESULTS.items()
]).sort_values('uplift@10', ascending=False).reset_index(drop=True)
results_df['rank'] = results_df['uplift@10'].rank(ascending=False).astype(int)

print('\n' + '='*80)
print('DL UPLIFT BENCHMARK  (5-fold OOF · uplift@10 binary · treatment×channel strat.)')
print('='*80)
print(results_df[['rank','model','uplift@10','ci80_lo','ci80_hi','qini_auc','elapsed_sec']]
      .to_string(index=False, float_format='{:.4f}'.format))
print('='*80)
results_df.to_csv(f'{ARTIFACTS_DIR}/dl_model_comparison.csv', index=False)
print(f'\n✅ Saved: {ARTIFACTS_DIR}/dl_model_comparison.csv')


## 11. Визуализация: uplift@10 и Uplift Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('DL Uplift Benchmark — OOF (binary buy/no-buy)', fontsize=14, fontweight='bold')

# Barchart + CI
ax = axes[0]
df_plot = results_df.sort_values('uplift@10')
color_map = {'TARNet': '#5c9ebe', 'CFR (Wasserstein)': '#6daa45',
             'CFR (MMD)': '#4a9e78', 'DragonNet': '#a86fdf',
             'RERUM (DragonNet)': '#e07b39'}
colors = [color_map.get(m, '#aaaaaa') for m in df_plot['model']]
y_pos = np.arange(len(df_plot))
ax.barh(y_pos, df_plot['uplift@10'], color=colors, height=0.6)
for i, (_, row) in enumerate(df_plot.iterrows()):
    ax.errorbar(row['uplift@10'], i,
                xerr=[[row['uplift@10']-row['ci80_lo']], [row['ci80_hi']-row['uplift@10']]],
                fmt='none', color='#333', capsize=4, linewidth=1.5)
ax.set_yticks(y_pos); ax.set_yticklabels(df_plot['model'], fontsize=9)
ax.set_xlabel('OOF uplift@10 (binary)'); ax.set_title('Model Ranking + 80% CI')
ax.axvline(0, color='black', lw=0.8, linestyle='--')

# Uplift curves
ax2   = axes[1]
y_bin = (y > 0).astype(int)
dl_scores = {'TARNet': oof_scores_tar, 'CFR (Wasserstein)': oof_scores_cfr_wass,
             'CFR (MMD)': oof_scores_cfr_mmd, 'DragonNet': oof_scores_dragon,
             'RERUM (DragonNet)': oof_scores_rerum}
palette = ['#5c9ebe','#6daa45','#4a9e78','#a86fdf','#e07b39']
for i, (name, scores) in enumerate(dl_scores.items()):
    try:
        n_arr, uplift_vals = uplift_curve(y_bin, scores, T)
        ax2.plot(n_arr, uplift_vals, label=name, color=palette[i], lw=1.8)
    except Exception as e:
        print(f'  Cannot plot {name}: {e}')
ax2.plot([0,1],[0,0],'k--',lw=0.8,label='Random')
ax2.set_xlabel('Fraction targeted'); ax2.set_ylabel('Uplift')
ax2.set_title('Uplift Curves (OOF)'); ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/dl_comparison_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {ARTIFACTS_DIR}/dl_comparison_plot.png')


## 12. Предсказания на тесте

Лучшая DL-модель по `uplift@10` обучается на **всём train** и экспортируется.


In [ ]:
dl_model_map = {
    'TARNet':            ('tarnet',    train_tarnet,    predict_tarnet),
    'CFR (Wasserstein)': ('cfr_wass',  lambda *a,**k: train_cfr(*a,ipm='wass',**k), predict_cfr),
    'CFR (MMD)':         ('cfr_mmd',   lambda *a,**k: train_cfr(*a,ipm='mmd', **k), predict_cfr),
    'DragonNet':         ('dragonnet', train_dragonnet, predict_dragonnet),
    'RERUM (DragonNet)': ('rerum',     train_rerum,     predict_rerum),
}

best_dl = max([n for n in dl_model_map if n in RESULTS],
              key=lambda n: RESULTS[n]['uplift@10'])
print(f'Best DL model: {best_dl}  (uplift@10={RESULTS[best_dl]["uplift@10"]:.4f})')

slug, train_fn, predict_fn = dl_model_map[best_dl]
X_tr_full, X_test_full = prepare_nn_test(train, test, FEAT_COLS)

print(f'Training {best_dl} on full train...')
t0         = time.time()
best_model = train_fn(X_tr_full, T.astype(np.float32), y.astype(np.float32),
                      in_dim=X_tr_full.shape[1])
test_preds = predict_fn(best_model, X_test_full)
print(f'  Done in {time.time()-t0:.0f}s')

submission = test[[ID_COL, COMM_COL]].copy()
submission['uplift_score'] = test_preds
submission.to_csv(f'{ARTIFACTS_DIR}/submission_{slug}.csv', index=False)
print(f'✅ Saved: {ARTIFACTS_DIR}/submission_{slug}.csv')
print(f'   uplift_score: mean={test_preds.mean():.4f}  std={test_preds.std():.4f}')


## 13. 🐛 Разбор бага в ноуте коллеги

### Почему RERUM показывал «очень низкие» метрики?

В ноуте коллеги (`pilot_modeling_rerum.ipynb`) смешаны две **несовместимые метрики** в одной таблице:

| Модель | Функция | Тип y | Шкала |
|--------|---------|-------|-------|
| TARNet / DragonNet | `uplift_at_k_manual` | непрерывный ₽ | ~16 ₽ |
| RERUM | `register_result` (sklift) | бинарный buy/no-buy | ~0.03 |

**~16 ₽** — это прирост выручки от топ-10% пользователей (рублёвая шкала).
**~0.03** — доля дополнительных покупателей (от 0 до 1).
Сравнивать их буквально — это как сравнивать метры и килограммы.

### Дополнительные технические баги:
1. **Отсутствие `StandardScaler`** — нейросети требуют нормализованного входа; без него оптимизация нестабильна
2. **Мало эпох (25 при warmup=5)** — ranking losses фактически работали лишь 20 эпох; недостаточно для сходимости
3. **`drop_last=False`** — батчи из 1-2 элементов ломают `BatchNorm1d` (деление на std ≈ 0)
4. **Нет `clip_grad_norm`** — начальные большие лоссы ZILN давали взрывные градиенты
5. **ZILN получал `log1p(y)` вместо `y`** — ZILN сам параметризует log-нормальное распределение,
   двойное логарифмирование искажает целевое распределение

### Все исправления применены в данном ноуте ✅


## 14. Экспорт всех артефактов

In [ ]:
dl_summary = {'results': RESULTS, 'best_dl_model': best_dl,
              'best_uplift10': float(RESULTS[best_dl]['uplift@10'])}
with open(f'{ARTIFACTS_DIR}/dl_summary.json', 'w') as f:
    json.dump(dl_summary, f, ensure_ascii=False, indent=2, default=str)

print('✅ DL artifacts saved:')
for fname in sorted(os.listdir(ARTIFACTS_DIR)):
    sz = os.path.getsize(f'{ARTIFACTS_DIR}/{fname}') / 1024
    print(f'  {fname:50s} {sz:.1f} KB')
print(f'\n🏆 Final leaderboard:')
print(results_df[['rank','model','uplift@10','qini_auc']]
      .to_string(index=False, float_format='{:.4f}'.format))
